# 05 Models: LambdaMART, collaborative features, neural ranking

**Purpose.** Train the rankers of PROJECT_PLAN section 5 (item 4) one model family at a time and measure every step against the same references on the same searches. This part sets up the training pipeline, profiles memory and trains the **deployable non-personalised base** (E14). Ablations, collaborative features and neural models follow in later sections.

**Primary benchmark.** Lagged history popularity (`pop_history_lag1d`, history actions at least one day old): 0.337 NDCG@10 on `validation_clean` (E13). Differences below 0.003-0.005 are not treated as meaningful.

**Rules (PROJECT_PLAN 5-6, FINDINGS 2-3).**
- Inputs only from the groups in `features_v3_meta.json`; never `session_id`, `wid`, `label` or `n_candidates`.
- Training searches: the 40,000-search dev subsample without searches with a clamped final gap (post-search actions in the history, E12), without an engaged candidate (every order scores the same) and without single-candidate lists.
- Early stopping on an **inner set** of 10% of these training searches (whole searches, seed 4222), so validation is used only to report, never to pick the number of trees.
- LightGBM LambdaRank with **linear gains** `label_gain=[0,1,2,3]` (LightGBM's default gains are exponential, E06) and NDCG@10 for early stopping; final numbers come from our exact tie-aware evaluator (`scripts/evaluation.py`) on complete validation lists.
- Main population `validation_clean` (clamped searches removed); all validation searches as sensitivity. Model comparisons are paired per-search differences with bootstrap intervals over searches.
- Candidates were shuffled within each search when the features were built (notebook 03), so file order, which encodes the label, cannot reach a model.
- Missing values (unknown metadata) stay missing: LightGBM learns where to send them, instead of treating unknown as "no match".

**How to run.** Locally after notebooks 02 and 03 (they write `data/processed/`). On Colab: open this notebook from GitHub (branch `BRANCH` below); the first cell clones the public repository and mounts Drive. Copy `search_table.parquet`, `split_v1.csv`, `features_v3_meta.json` and the two `features_v3_*.parquet` files to `MyDrive/BT4222 Group Project/processed/` first. CPU is enough for this part.

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/liamkunert/BT4222-group17-jdsearch.git", "/content/BT4222-group17-jdsearch"],
                   check=True)
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
else:
    PROCESSED = REPO_ROOT / "data/processed"

sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import RESULTS, evaluate, log_result, paired_difference, summarise

pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v3"
SEED = 4222        # data decisions (inner split)
MODEL_SEED = 1     # LightGBM; three seeds only for finalists
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]
ALL_FEATURES = [f for group in meta["groups"].values() for f in group]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES)) == 97
assert not set(ALL_FEATURES) & {"session_id", "wid", "label", "n_candidates"}
print(f"LightGBM {lgb.__version__}; {len(ALL_FEATURES)} features in {len(meta['groups'])} groups; processed data: {PROCESSED}")

## 1. Training population and inner early-stopping set

In [ ]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
drops = pd.DataFrame({
    "clamped final gap": dev_info.final_gap_clamped,
    "no engaged candidate": dev_info.n_engaged == 0,
    "single candidate": dev_info.n_candidates == 1,
})
train_pool = dev_ids[~drops.any(axis=1).to_numpy()]
rng = np.random.default_rng(SEED)
inner_ids = np.sort(rng.choice(train_pool.to_numpy(), size=round(0.1 * len(train_pool)), replace=False))
fit_ids = np.setdiff1d(train_pool.to_numpy(), inner_ids)
print(f"dev searches {len(dev_ids):,}; kept for training {len(train_pool):,} "
      f"(fit {len(fit_ids):,}, inner early-stopping set {len(inner_ids):,})")
drops.sum().rename("dropped searches (overlapping)").to_frame()

## 2. Loading and memory profile

Features are read **column by column into one pre-allocated float32 matrix** (column-major, which LightGBM accepts without copying), so no intermediate DataFrame of the full table is built. The rows of each search are contiguous in the feature files; LightGBM's `group` is the list of consecutive search sizes.

The profile loads all 97 features for the dev subsample and validation and builds a LightGBM dataset from them, which is what the full-model ablations need. "Peak" is the highest memory the process has used so far.

In [ ]:
PROCESS = psutil.Process()


def memory_gb():
    """(current, peak) resident memory of this process in GB."""
    info = PROCESS.memory_info()
    peak = getattr(info, "peak_wset", None)  # Windows
    if peak is None:  # Linux / Colab: ru_maxrss is in KiB
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    return info.rss / 1e9, peak / 1e9


def load_features(split, columns, sessions=None):
    """Session ids, float32 feature matrix (column-major) and labels; rows restricted to `sessions` if given."""
    path = PROCESSED / f"features_{FEATURES}_{split}.parquet"
    ids = pq.read_table(path, columns=["session_id"]).column(0).to_numpy()
    if not (np.diff(ids) >= 0).all():
        raise ValueError("Rows must be sorted by session_id so that every search is contiguous")
    keep = np.ones(len(ids), dtype=bool) if sessions is None else np.isin(ids, sessions)
    matrix = np.empty((int(keep.sum()), len(columns)), dtype=np.float32, order="F")
    for j, column in enumerate(columns):
        matrix[:, j] = pq.read_table(path, columns=[column]).column(0).to_numpy()[keep]
    labels = pq.read_table(path, columns=["label"]).column(0).to_numpy()[keep]
    return ids[keep], matrix, labels


def group_sizes(ids):
    """Sizes of consecutive runs of equal session ids (LightGBM's `group`)."""
    bounds = np.r_[0, np.flatnonzero(np.diff(ids)) + 1, len(ids)]
    return np.diff(bounds)

In [ ]:
profile = [("start", *memory_gb())]
start = time.time()
train_sid, train_X, train_y = load_features("train_dev", ALL_FEATURES, sessions=train_pool)
profile.append(("train features loaded (97 columns)", *memory_gb()))
categorical = [ALL_FEATURES.index(c) for c in meta["categorical"]]
probe = lgb.Dataset(train_X, label=train_y, group=group_sizes(train_sid), categorical_feature=categorical,
                    free_raw_data=False, params={"verbose": -1, "feature_pre_filter": False}).construct()
profile.append(("LightGBM dataset built", *memory_gb()))
valid_sid, valid_X, valid_y = load_features("validation", ALL_FEATURES)
profile.append(("validation features loaded", *memory_gb()))
seconds = time.time() - start
train_rows, train_gb = train_X.shape[0], train_X.nbytes / 1e9
del probe, train_X, valid_X, train_sid, valid_sid, train_y, valid_y
gc.collect()
profile.append(("all freed", *memory_gb()))
profile = pd.DataFrame(profile, columns=["step", "current_gb", "peak_gb"]).set_index("step")
full_rows = train_rows * (assignment.split == "train").sum() / len(dev_ids)
print(f"training rows {train_rows:,} ({train_gb:.2f} GB as float32); load + build {seconds:.0f} s")
print(f"estimate for all training searches: {full_rows / 1e6:.1f}M rows, "
      f"{full_rows * len(ALL_FEATURES) * 4 / 1e9:.1f} GB as float32 before LightGBM's dataset")
print(f"machine memory: {psutil.virtual_memory().total / 1e9:.1f} GB total")
profile

## 3. E14: deployable non-personalised base (G0 + G1 + G2a)

Ten features: query length (G0, acts only through interactions because it is constant within a list), the eight query-title relevance features (G1) and lagged history popularity (G2a). Nothing in the base uses the shopper's history or same-day information, so it is the trained version of the deployable reference.

One fixed LightGBM configuration is used for all ablations (PROJECT_PLAN 5a); it is not tuned here. Success criterion, fixed before running: the lower bound of the paired 95% interval against lagged popularity on `validation_clean` is above +0.003.

In [ ]:
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [10],
    "label_gain": [0, 1, 2, 3],          # linear gains, as in our evaluator
    "lambdarank_truncation_level": 20,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_data_in_leaf": 500,
    "feature_pre_filter": False,
    "seed": MODEL_SEED,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
MAX_ROUNDS, PATIENCE = 3000, 100
BASE = meta["groups"]["G0_search"] + meta["groups"]["G1_relevance"] + meta["groups"]["G2a_popularity_deployable"]


def train_ranker(columns, params=PARAMS, label=""):
    """Fit on the fit searches, stop early on the inner set; return the booster and a summary."""
    cats = [columns.index(c) for c in meta["categorical"] if c in columns]
    sid, X, y = load_features("train_dev", columns, sessions=fit_ids)  # loaded separately: no copies of X
    fit_set = lgb.Dataset(X, label=y, group=group_sizes(sid), feature_name=columns, categorical_feature=cats)
    sid_inner, X_inner, y_inner = load_features("train_dev", columns, sessions=inner_ids)
    inner_set = lgb.Dataset(X_inner, label=y_inner, group=group_sizes(sid_inner), reference=fit_set)
    start = time.time()
    booster = lgb.train(params, fit_set, num_boost_round=MAX_ROUNDS, valid_sets=[inner_set], valid_names=["inner"],
                        callbacks=[lgb.early_stopping(PATIENCE, verbose=False), lgb.log_evaluation(200)])
    summary = {"model": label, "features": len(columns), "best_iteration": booster.best_iteration,
               "inner_ndcg_at_10": booster.best_score["inner"]["ndcg@10"], "train_seconds": round(time.time() - start),
               "peak_memory_gb": round(memory_gb()[1], 2)}
    del X, X_inner, fit_set, inner_set
    gc.collect()
    return booster, summary


def score_validation(booster, columns):
    """Long table session_id, label, score for all validation candidates."""
    sid, X, y = load_features("validation", columns)
    frame = pd.DataFrame({"session_id": sid, "label": y,
                          "score": booster.predict(X, num_iteration=booster.best_iteration)})
    del X
    return frame


base_model, base_summary = train_ranker(BASE, label="LambdaMART base G0+G1+G2a")
pd.Series(base_summary).to_frame("E14 base")

In [ ]:
scored = score_validation(base_model, BASE)
scored["pop_lag1d"] = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                                    columns=["pop_history_lag1d"]).column(0).to_numpy()
per_search = {
    "LambdaMART base": evaluate(scored, "score", expected_sizes=validation_sizes),
    "lagged history popularity": evaluate(scored.assign(ref=scored.pop_lag1d.fillna(0).astype(float)), "ref",
                                          expected_sizes=validation_sizes),
    "random order": evaluate(scored.assign(constant=0.0), "constant", expected_sizes=validation_sizes),
}
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]

# Sanity check: the reference reproduces its registry value (E13, validation_clean).
registry = pd.read_csv(RESULTS)
logged_ref = registry.query("experiment == 'E13' and model == 'history popularity, lag 1 day' "
                            "and split == 'validation_clean'").ndcg_at_10.iloc[0]
assert abs(per_search["lagged history popularity"].ndcg.loc[clean_ids].mean() - logged_ref) < 1e-4

rows = {}
for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
    for name, per in per_search.items():
        row = summarise(per.loc[ids]).iloc[0].to_dict()
        if name != "lagged history popularity":
            d = paired_difference(per.loc[ids], per_search["lagged history popularity"].loc[ids])
            row.update(vs_lagged_pop=d[0], vs_lagged_pop_low=d[1], vs_lagged_pop_high=d[2])
        rows[(population, name)] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

### By segment (validation_clean)

Popularity is much weaker in long, deep-scroll lists (E12), so list length is the segment to watch; activity in the last 7 days separates shoppers with fresh history from returning ones. Each cell is the paired difference base minus lagged popularity within the segment.

In [ ]:
info = searches.loc[clean_ids]
segments = {
    "list length": pd.cut(info.n_candidates, [0, 60, 84, 95, 200], labels=["1-60", "61-84", "85-95", "96-200"]),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=info.index),
}
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search["LambdaMART base"].loc[ids], per_search["lagged history popularity"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

Split importance (share of total gain). This shows what the trees use, not how much shoppers value a signal; correlated features share gain arbitrarily.

In [ ]:
importance = pd.Series(base_model.feature_importance("gain"), index=BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

### Log to the results registry

Experiment E14, one row per population. Re-running the notebook does not duplicate rows.

In [ ]:
NOTES = (f"LightGBM lambdarank, features {FEATURES} G0+G1+G2a, linear gains, trunc 20, 31 leaves, lr 0.05, "
         f"min leaf 500, inner early stopping (best iteration {base_summary['best_iteration']}), seed {MODEL_SEED}")
logged = pd.read_csv(RESULTS)
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    if ((logged.experiment == "E14") & (logged.model == "LambdaMART base G0+G1+G2a") & (logged.split == split)).any():
        continue
    log_result("E14", "LambdaMART base G0+G1+G2a", split, per_search["LambdaMART base"].loc[ids], notes=NOTES)
pd.read_csv(RESULTS).query("experiment == 'E14'")